In [ ]:
import glob, re, joblib
import numpy as np
import scanpy as sc
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import json, re
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from statannotations.Annotator import Annotator
import string
from XplainTDA_PH import XplainTDA
import logomaker
import pyarrow.parquet as pq
import matplotlib.patches as mpatches
from Utils import colors_to_use, colors_to_use_pastel
from sklearn.metrics import (
    adjusted_mutual_info_score,
    normalized_mutual_info_score)
import math
from Utils import *

# Figure 2

In [ ]:
datasets = ["PBMC", "HumanBrain","BMMCMultiOme"]
omics = ["GEX", "Peak"]
dims = [0, 1, 2]
distances = ["distance_by_diameter"]
q=0.95

def read_results(pattern):
    """Read every existing result file and infer dataset/omic from its path."""
    out = []
    for path in Path("Datasets").glob(pattern):
        tmp = pd.read_csv(path)
        parts = path.parts
        i = parts.index("Datasets")
        tmp["dataset"], tmp["omic"] = parts[i + 1], parts[i + 3]
        out.append(tmp)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def k_value(x):
    m = re.search(r"k=(\d+)", str(x))
    return np.inf if x == "full" else int(m.group(1)) if m else np.nan


def knn_curve(sub, dim):
    """H0 uses the full graph; H1/H2 use the largest k as reference."""
    if sub.empty:
        return sub

    labels = pd.unique(sub[["label_a", "label_b"]].to_numpy().ravel())
    ks = [x for x in labels if str(x).startswith("k=")]
    ref = "full" if dim == 0 else max(ks, key=k_value, default=None)
    if ref is None:
        return pd.DataFrame()

    a, b = sub["label_a"].astype(str), sub["label_b"].astype(str)
    out = sub[
        ((a == ref) & b.str.startswith("k="))
        | ((b == ref) & a.str.startswith("k="))
    ].copy()

    out["k"] = np.where(
        out["label_a"] == ref,
        out["label_b"].map(k_value),
        out["label_a"].map(k_value),
    )
    return out.sort_values("k").drop_duplicates("k")


knn = read_results(
    "*/XplainTDA/*/Stability/GRAE/kNN/quantile_bottleneck_metrics.csv"
)
sub = read_results(
    "*/XplainTDA/*/Stability/GRAE/Subsampling/pairwise_quantile_bottleneck.csv"
)

for frame in (knn, sub):
    if not frame.empty:
        frame.query(
            "dataset in @datasets and omic in @omics "
            "and homology_dim in @dims",
            inplace=True,
        )
        frame.drop(frame.index[~np.isclose(frame["quantile"], q)], inplace=True)

datasets = [
    d for d in datasets
    if (not knn.empty and d in knn["dataset"].values)
    or (not sub.empty and d in sub["dataset"].values)
]
omics = [
    o for o in omics
    if (not knn.empty and o in knn["omic"].values)
    or (not sub.empty and o in sub["omic"].values)
]

BASE = Path("Datasets")

def numeric_label(value, pattern, default=np.nan):
    """Extract an integer from a string label."""
    match = re.search(pattern, str(value))
    return int(match.group(1)) if match else default


def read_results(pattern, numeric_columns):
    """Read matching result files and add path metadata and numeric labels."""
    frames = []

    for path in sorted(BASE.glob(pattern)):
        parts = path.relative_to(BASE).parts
        df = pd.read_csv(path).assign(
            dataset=parts[0],
            omic=parts[2],
            source_file=str(path),
        )

        for output, (column, regex, default) in numeric_columns.items():
            df[output] = df[column].map(
                lambda x: numeric_label(x, regex, default)
            )

        frames.append(df)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


df_k = read_results(
    "*/XplainTDA/*/Stability/GRAE/kNN/quantile_bottleneck_metrics.csv",
    {
        "k1_numeric": ("label_a", r"k=(\d+)", np.inf),
        "k2_numeric": ("label_b", r"k=(\d+)", np.inf),
    },
)

df_s = read_results(
    "*/XplainTDA/*/Stability/GRAE/Subsampling/"
    "pairwise_quantile_bottleneck.csv",
    {
        "subset_a_id": ("subset_a", r"subset_(\d+)", np.nan),
        "subset_b_id": ("subset_b", r"subset_(\d+)", np.nan),
    },
)

In [ ]:
datasets_aliases = {"PBMC" : "PBMC", "HumanBrain" : "Brain", "BMMCMultiOme" : "BMMC"}
distance="distance_by_diameter"
omic_colors = {"GEX": colors_to_use[19], "Peak": colors_to_use[16]}
omic_labels = {"GEX": "GEX", "Peak": "ATAC"}
dataset_styles = {"PBMC": "-", "HumanBrain": "--", "BMMCMultiOme" : ":"}
# New outputs are in long format: select q=0.95 first.
df_k = df_k.loc[np.isclose(df_k["quantile"], q)].copy()
df_s = df_s.loc[np.isclose(df_s["quantile"], q)].copy()

def reference_distances(sub, dim):
    if sub.empty:
        return pd.DataFrame()

    labels = sorted(
        [
            x for x in set(sub["label_a"]) | set(sub["label_b"])
            if str(x).startswith("k=")
        ],
        key=k_value,
    )
    if not labels:
        return pd.DataFrame()

    ref = "full" if dim == 0 else labels[-1]
    out = sub.loc[
        ((sub["label_a"] == ref) & sub["label_b"].isin(labels))
        | ((sub["label_b"] == ref) & sub["label_a"].isin(labels))
    ].copy()

    out["k"] = np.where(
        out["label_a"] == ref,
        out["label_b"].map(k_value),
        out["label_a"].map(k_value),
    )

    if dim:
        row = sub.iloc[0][["dataset", "omic", "homology_dim", "quantile"]].to_dict()
        row.update(
            label_a=ref,
            label_b=ref,
            k=k_value(ref),
            **{distance: 0.0},
        )
        out = pd.concat([out, pd.DataFrame([row])], ignore_index=True)

    return out.sort_values("k")


def format_ax(ax, panel, dim, xlabel="", log_y=False):
    ax.set(xlabel=xlabel, ylabel=distance,
           yscale="log" if log_y else "linear")
    ax.set_title(
        rf"$\mathbf{{{string.ascii_uppercase[panel]}}}$     H{dim}",
        loc="left", x=-.08, y=1.1 if panel >= 3 else 1.075,
        fontsize=20, pad=0,
    )
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(False)


labels = set(df_k["label_a"]) | set(df_k["label_b"])
ks = sorted({
    int(k_value(x))
    for x in labels
    if str(x).startswith("k=") and np.isfinite(k_value(x))
})

df_s["omic_label"] = df_s["omic"].map(omic_labels)
palette = {omic_labels[o]: omic_colors[o] for o in omics}

params = {
    'axes.labelsize': 17,
    'axes.titlesize': 17,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    "lines.linewidth": 3,
    "figure.dpi": 300}
plt.rcParams.update(params)
fig, axes = plt.subplots(2,3, figsize=(16,10))

# kNN stability
for col, dim in enumerate(dims):
    ax = axes[0, col]

    for dataset in datasets:
        for omic in omics:
            sub = df_k.loc[
                (df_k["dataset"] == dataset)
                & (df_k["omic"] == omic)
                & (df_k["homology_dim"] == dim)
            ]
            line = reference_distances(sub, dim)

            if not line.empty:
                ax.plot(
                    line["k"], line[distance],
                    color=omic_colors[omic],
                    linestyle=dataset_styles[dataset],
                    marker="o", markersize=8,
                )

    format_ax(ax, col, dim, "k")
    ax.set_xscale("log")
    ax.set_ylabel("q-Bottleneck loss")
    ax.set_xticks(ks, ks, rotation=90)

# Subsampling stability
for col, dim in enumerate(dims):
    ax = axes[1, col]
    sub = df_s.loc[df_s["homology_dim"] == dim]

    if not sub.empty:
        sns.stripplot(
            data=sub,
            y="dataset", x=distance, hue="omic_label",
            order=datasets,
            hue_order=list(omic_labels.values()),
            palette=palette,
            dodge=True, jitter=.18, size=8, alpha=.8,
            ax=ax,
        )
        if ax.get_legend():
            ax.get_legend().remove()

    format_ax(ax, col + 3, dim)
    ax.set_xscale("log")
    ax.set_xlim(1e-5, 1e0)
    ax.set_xticks(10.0 ** np.arange(-5, 1))
    ax.set_xlabel("q-Bottleneck loss")
    ax.set_ylabel("Data set")
    ax.set_yticks(range(0,3), datasets_aliases.values())

dataset_handles = [
    Line2D([], [], color="black",
           linestyle=dataset_styles[d], label=a)
    for d, a in datasets_aliases.items()
]
omic_handles = [
    mpatches.Patch(color=omic_colors[o], label=omic_labels[o])
    for o in omics
]

for handles, title, anchor in [
    (dataset_handles, "Data set", (1.10, .80)),
    (omic_handles, "Omic", (1.10, .42)),
]:
    fig.legend(
        handles=handles, title=title,
        bbox_to_anchor=anchor, loc="upper right",
        fontsize=17, title_fontsize=20, frameon=False,
    )
plt.tight_layout()
fig.savefig("Figures/XplainTDA_Figure2.png", dpi=300, bbox_inches="tight", pad_inches=.3)
plt.show()

# Figure 3

In [ ]:
datasets_aliases = {"PBMC": "PBMC", "HumanBrain": "Brain", "BMMCMultiOme": "BMMC"}
PLOT_LABELS = {"PBMC": "Raw_CellType", "HumanBrain": "Raw_celltype",
               "BMMCMultiOme": "Coarse_CellType"}
datasets, omic, k, MIN_N_CELLS_IN_CC = list(datasets_aliases), "GEX", 75, 5
homology_palette = {"H0": colors_to_use[5], "H1": colors_to_use[14], "H2": colors_to_use[11]}

params = {
    'axes.labelsize': 17,
    'axes.titlesize': 17,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    "lines.linewidth": 3,
    "figure.dpi": 300}
plt.rcParams.update(params)

def read_annotation_output(dataset, label, min_cc_size=MIN_N_CELLS_IN_CC):
    out = Path("Datasets") / dataset / "XplainTDA" / omic / "Annotation" / label
    merged, best = pd.read_csv(out / "combined_results.csv"), pd.read_csv(out / "best_filtration.csv").iloc[0]
    cc, annotation = merged.set_index("Barcode")["clusterID"], merged.set_index("Barcode")["Celltype"]
    annotated = annotation.notna() & annotation.astype(str).str.strip().ne("")
    removed = set(cc.loc[~annotated].dropna()) | set(cc.value_counts()[lambda x: x < min_cc_size].index)
    retained = cc.notna() & annotated & ~cc.isin(removed)
    return cc, annotation, retained, best

tdas = {d: XplainTDA.load(Path("Datasets") / d / "XplainTDA" / omic / "Stability" / "GRAE" / "kNN" / "tda_objects" / f"tda_knn_k{k}.joblib") for d in datasets}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
color_idx = 0

# Persistence diagrams
for panel, (dataset, ax) in enumerate(zip(datasets, axes[0])):
    dgms = tdas[dataset].dgms[:3]
    finite = [np.asarray(d)[np.isfinite(d).all(axis=1)] for d in dgms]
    lim = max(d.max() for d in finite if len(d)) * 1.05

    for h, dgm in enumerate(finite):
        ax.scatter(dgm[:, 0], dgm[:, 1], s=18, color=homology_palette[f"H{h}"], edgecolor="none", label=rf"$H_{h}$")

    ax.plot([0, lim], [0, lim], "--", color="grey", lw=1.5, zorder=0)
    ax.set_title(rf"$\mathbf{{{string.ascii_uppercase[panel]}}}$ {datasets_aliases[dataset]}",
                 loc="left", x=-.05, y=1.03, fontsize=18, pad=0)
    ax.set(xlabel="Birth", xlim=(-.025, lim), ylim=(-.025, lim))
    ax.set_aspect("equal")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(False)

axes[0, 0].set_ylabel("Death")
handles, labels = axes[0, 0].get_legend_handles_labels()
axes[0, -1].legend(handles, labels, bbox_to_anchor=(1.02, .5), markerscale=2.5,
                   loc="center left", fontsize=17, title_fontsize=20, frameon=False)

# Connected-component annotation
for panel, (dataset, ax) in enumerate(zip(datasets, axes[1]), start=3):
    cc, annotation, retained, best = read_annotation_output(dataset, PLOT_LABELS[dataset])
    counts = pd.crosstab(cc.loc[retained], annotation.loc[retained].astype(str)).sort_index()
    panel_labels = counts.columns.tolist()

    if color_idx + len(panel_labels) > len(colors_to_use_pastel):
        raise ValueError(f"Need at least {color_idx + len(panel_labels)} colours.")

    palette = dict(zip(panel_labels, colors_to_use_pastel[color_idx:color_idx + len(panel_labels)]))
    color_idx += len(panel_labels)
    x, bottom = np.arange(len(counts)), np.zeros(len(counts))

    for celltype in panel_labels:
        values = counts[celltype].to_numpy()
        ax.bar(x, values, bottom=bottom, color=palette[celltype], edgecolor="none")
        bottom += values

    ax.set_xticks(x, range(len(x)), rotation=90)
    ax.set_title(rf"$\mathbf{{{string.ascii_uppercase[panel]}}}$ {datasets_aliases[dataset]}"
                 f"\nAMI: {best['ami']:.2f}",
                 loc="left", x=-.05, y=1.03, fontsize=18, pad=0)
    ax.set(xlabel="Connected component", ylabel="Number of cells", yscale="log")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(False)

    x_leg = .55 if dataset == "HumanBrain" else .85
    ax.legend(handles=[Patch(color=palette[c], label=c) for c in panel_labels], markerscale=250,
              bbox_to_anchor=(x_leg, 1.1), loc="upper left", frameon=False, fontsize=10)

    ami = adjusted_mutual_info_score(annotation.loc[retained].astype(str),
                                     cc.loc[retained].astype(str))
plt.tight_layout()
fig.savefig("Figures/XplainTDA_Figure3.png", dpi=300, bbox_inches="tight", pad_inches=.3)
plt.show()

In [ ]:
omic = "GEX"
MIN_N_CELLS_IN_CC = 5

TABLE_LABELS = {
    "PBMC": ["Raw_CellType"],
    "HumanBrain": ["Raw_celltype"],
    "BMMCMultiOme": ["Coarse_CellType"],
}

def get_cc_annotation(dataset, label, min_cc_size=MIN_N_CELLS_IN_CC):
    out_dir = (
        Path("Datasets")
        / dataset
        / "XplainTDA"
        / omic
        / "Annotation"
        / label
    )

    results = pd.read_csv(out_dir / "combined_results.csv")
    best = pd.read_csv(out_dir / "best_filtration.csv").iloc[0]

    cc = results.set_index("Barcode")["clusterID"]
    annotation = results.set_index("Barcode")["Celltype"]
    filtration = best["filtration_value"]
    ami = best["ami"]

    annotated = annotation.notna() & annotation.astype(str).str.strip().ne("")

    cc_sizes = cc.value_counts(dropna=True)
    cc_unannotated = set(cc.loc[~annotated].dropna().unique())
    cc_small = set(cc_sizes[cc_sizes < min_cc_size].index)
    removed_cc = cc_unannotated | cc_small

    retained = cc.notna() & annotated & ~cc.isin(removed_cc)

    return filtration, cc, annotation, retained, {
        "removed_cc_unannotated": len(cc_unannotated),
        "removed_cc_small": len(cc_small),
        "removed_cc_total": len(removed_cc),
        "removed_cells_total": int(cc.isin(removed_cc).sum())}, ami


rows = []

for dataset, labels in TABLE_LABELS.items():
    for label in labels:
        filtration, cc, annotation, retained, stats, ami = get_cc_annotation(
            dataset, label
        )

        rows.append({
            "Data set": dataset,
            "Filtration": filtration,
            "AMI": ami,
            "Number of removed cells": stats["removed_cells_total"],
            "Number of removed CC": stats["removed_cc_total"]})

score_table = pd.DataFrame(rows)
score_table

In [ ]:
score_table.to_csv("Tables/Table_1.tsv", sep="\t")

# Figure 4

In [ ]:
pbmc = {
    "Monocytes": ["FCN1","FCNM","CD14","TCF7L2","TCF4","TCF-4","FCGR3A","CD16A","CD16","FCGR3","FCG3","IGFR3","IMD20","CD16-II","FCRIIIA","FcGRIIIA","LYN","JTK8","SAIDV","p53Lyn","p56Lyn"],
    "DC": ["CLEC9A","CADM1","CLEC10A","FCER1A","CST3","COTL1","LYZ","DMXL2","GZMB","IL3RA","COBLL1","TCF4"],
    "NK": ["GNLY","NKG7","CD247","GRIK4","FCER1G","TYROBP","KLRG1","FCGR3A","ID2","PLCG2","SYNE1"],
    "B": ["MS4A1","IL4R","IGHD","FCRL1","IGHM","SSPN","ITGB1","EPHA4","COL4A4","PRDM1","ZNF215","IRF4","CD38","XBP1","PAX5","BCL11A","BLK","MME","CD24","ACSM3","MSI2","MZB1","HSP90B1","FNDC3B","IGKC","JCHAIN"],
    "T": ["CD4","IL7R","TRBC2","ITGB1","CCR7","CD8A","CD8B","GZMB","GZMA","CCL5","GZMK","GZMH","CD69","CD38","LEF1","TCF7"],
}
human_brain = {'Purkinje_neuron': ['PCP2', 'TRABD2B', 'ZNF385C', 'ATP2A3', 'IL20RA', 'GNG13', 'CA8', 'CALB1', 'ITPR1', 'FOXP2'], 
                 'Inhibitory_neuron': ['OTX2', 'DMBX1', 'SST', 'TFAP2B', 'LHX6', 'CXCL14', 'CASR', 'MYO5B', 'ANO1', 'EYA4', 'VIP', 'HAPLN1', 'LRRC38', 'PTHLH', 'TFAP2A', 'SOX14', 'SCGN', 'TAC3', 'CHRDL1', 'SP8', 'DCN', 'WNT16', 'TCIM', 'SFRP1', 'LIPG', 'ADGRG6', 'BHLHE22', 'PAX2', 'RGS5', 'NMU', 'C1QL1', 'PCDH18', 'ELFN1', 'KMO', 'PRDM1', 'MCUB', 'CRH', 'CRHBP', 'PAWR', 'TNFSF15', 'GATA3', 'SLC6A5', 'EDNRA', 'HTR3A', 'RNF144B', 'SCARA5', 'NDNF', 'NPR3', 'IQGAP2', 'MSR1', 'LMCD1', 'PDGFD', 'RELN', 'NPPC', 'SLC22A3', 'CHST9', 'GRPR', 'COL15A1', 'NPSR1', 'OSTN', 'PTGFR', 'TAC1', 'HGF', 'MAFB', 'SLC17A8', 'FAM89A', 'KCNJ5', 'ITGA1', 'PNOC', 'KCNS3', 'CPED1', 'NGF', 'CMTM8', 'ANGPT1', 'MYB', 'NPY', 'EREG', 'PROM1', 'MEPE', 'IL18R1', 'HPGD', 'SHISA8', 'MYH11', 'CASP4LP', 'DDR2', 'SULF1', 'SCTR', 'HGFAC', 'ADAMTS17', 'MYBPC1', 'NOG', 'TMEM255A', 'CYP26A1', 'KIT', 'NR2E1', 'CLCNKA', 'FBN2', 'TSPAN12', 'PDCL2', 'XDH'], 
                 'Astrocyte': ['FGFR3', 'GFAP', 'TNC', 'SLC14A1', 'GJA1', 'ETNPPL', 'CD38', 'CD44', 'ID3', 'GLI3', 'AQP4', 'F3', 'EFEMP1', 'LGR6', 'SDC4', 'ANGPTL4', 'OAF', 'PAPLN', 'GJB6', 'PAX3', 'ITPRID1', 'NAA11', 'LCAT', 'ZNF98', 'CPAMD8', 'MLC1', 'HPSE2', 'SLC7A10', 'AGT', 'RASL12', 'SMTN', 'MT1G', 'AEBP1', 'EDNRB', 'EMX2', 'PYGM', 'RANBP3L'], 
                 'Oligodendrocyte': ['OPALIN', 'CD9', 'MAG', 'CNDP1', 'RNASE1', 'MAL', 'SLCO1A2', 'FOLH1', 'NKX6-2', 'PLLP', 'MBOAT1', 'PTCSC3', 'SLC5A11', 'ITGA2', 'S100A1', 'MUSK', 'MOG', 'ASPA', 'CARNS1', 'PPP1R14A', 'LRP2', 'S100B', 'SH3TC2', 'SPATA22', 'TMEM98'], 
                 'Microglia': ['CD74', 'LNCAROD', 'HLA-DRA', 'CX3CR1', 'FYB1', 'C3', 'CSF1R', 'DOCK8', 'SYK', 'SRGN', 'RGS1', 'ITGAX', 'C1QC', 'PTPRC', 'AIF1', 'CCDC26', 'TTR', 'GPR183', 'CD69', 'HLA-DRB1', 'C1QB', 'OLR1', 'CD84', 'HLA-DPA1', 'TNFRSF1B', 'TYROBP', 'IFI30', 'MS4A6A', 'PIK3R5', 'CXCR4', 'HLA-DQA1', 'HLA-DRB5'],
                "Molec_Layer_Interneur": ["GAD1", "GAD2", "SLC32A1", "SLC6A1","SORCS3", "NXPH1", "GJD2","PVALB"]}

bmmc_multiome = {
    "Monocyte": ["FCN1","CD14","TCF7L2","FCGR3A","LYN"],
    "Progenitor": ["CD14","ID2","VCAN","S100A9","CLEC12A","KLF4","PLAUR","IGLL1","VPREB1","MME","EBF1","SSBP2","BACH2","CD79B","IGHM","PAX5","PRKCE","DNTT","MPO","BCL2","KCNQ5","CSF3R","PRTN3","NRIP1","MECOM","PROM1","CD34","NKAIN2","ZNF385D","ITGA2B","RYR3","PLCB1"],
    "DC": ["CLEC9A","CADM1","CLEC10A","FCER1A","CST3","COTL1","LYZ","DMXL2","GZMB","IL3RA","COBLL1","TCF4"],
    "Erythroid": ["SLC4A1","SLC25A37","HBB","HBA2","HBA1","TFRC","MKI67","CDK6","SYNGR1","HBM","GYPA"],
    "NK/ILC": ["GNLY","NKG7","CD247","GRIK4","FCER1G","TYROBP","KLRG1","FCGR3A","ID2","PLCG2","SYNE1"],
    "B": ["MS4A1","IL4R","IGHD","FCRL1","IGHM","SSPN","ITGB1","EPHA4","COL4A4","PRDM1","ZNF215","IRF4","CD38","XBP1","PAX5","BCL11A","BLK","MME","CD24","ACSM3","MSI2","MZB1","HSP90B1","FNDC3B","IGKC","JCHAIN"],
    "T": ["CD4","IL7R","TRBC2","ITGB1","CCR7","CD8A","CD8B","GZMB","GZMA","CCL5","GZMK","GZMH","CD69","CD38","LEF1","TCF7"],
}

markers_by_dataset = {
    "PBMC": pbmc,
    "HumanBrain": human_brain,
    "BMMCMultiOme": bmmc_multiome,
}

In [ ]:
datasets = ["PBMC", "HumanBrain", "BMMCMultiOme"]
labels = {
    "PBMC": "Raw_CellType",
    "HumanBrain": "Raw_celltype",
    "BMMCMultiOme": "Coarse_CellType",
}
omics = ["GEX"]

def read_json(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else None

def result_outdir(dataset, omic):
    return Path(f"Datasets/{dataset}/XplainTDA/{omic}/Explanations/{labels[dataset]}")

def load_one_result(dataset, omic, load_heavy=False):
    outdir = result_outdir(dataset, omic)
    exp = outdir / "results" / "explanations.csv"
    if not exp.exists():
        return None

    groups_dir = outdir / "results" / "groups"
    emb_dir = outdir / "embedding"
    tda_dir = outdir / "tda"
    emb_files = sorted(emb_dir.glob("*.npy"))

    out = {
        "dataset": dataset,
        "omic": omic,
        "label": labels[dataset],
        "outdir": outdir,
        "explanations": pd.read_csv(exp),
        "groups": {p.stem: pd.read_csv(p) for p in sorted(groups_dir.glob("*.csv"))},
        "skipped": pd.read_csv(outdir / "results" / "skipped_groups.csv")
                   if (outdir / "results" / "skipped_groups.csv").exists()
                   else pd.DataFrame(columns=["group", "reason"]),
        "config": read_json(outdir / "config.json"),
        "embedding_metadata": read_json(emb_dir / "metadata.json"),
    }

    if load_heavy:
        out["embedding"] = np.load(emb_files[0]) if emb_files else None
        out["tda_states"] = {p.stem: joblib.load(p) for p in sorted(tda_dir.glob("*.joblib"))}

    return out

xplain_results = {}

for dataset in datasets:
    for omic in omics:
        file = Path(f"Datasets/{dataset}/CM/{dataset}_{omic}_Def.h5ad")
        if not file.is_file():
            print(f"Missing AnnData: {file}")
            continue

        loaded = load_one_result(dataset, omic)
        if loaded is None:
            print(f"Missing results: {result_outdir(dataset, omic)}")
            continue

        xplain_results[(dataset, omic)] = loaded

if not xplain_results:
    raise FileNotFoundError("No XplainTDA results were loaded.")

combined_explanations = pd.concat(
    [
        r["explanations"].assign(
            dataset=dataset,
            omic=omic,
            label=r["label"],
            outdir=str(r["outdir"]),
        )
        for (dataset, omic), r in xplain_results.items()
    ],
    ignore_index=True,
)

combined_explanations["dataset"] = combined_explanations["dataset"].replace({"Human_Brain": "HumanBrain"})
combined_explanations["group"] = combined_explanations["group"].astype(str)
combined_explanations["gene"] = combined_explanations["gene"].astype(str)
combined_explanations["homology_dim"] = combined_explanations["homology_dim"].astype(int)

In [ ]:
def make_marker_random_df(df, markers_by_dataset, random_factor=1, seed=3):
    rng = np.random.default_rng(seed)
    rows = []

    for (dataset, omic, h, celltype), sub in df.groupby(
        ["dataset", "omic", "homology_dim", "group"], sort=False
    ):
        marker_dict = markers_by_dataset.get(dataset, {})
        marker_set = set(marker_dict.get(celltype, []))
        if not marker_set:
            continue

        all_markers = set().union(*(set(v) for v in marker_dict.values()))
        marker_rows = sub[sub["gene"].isin(marker_set)].copy()
        if marker_rows.empty:
            continue

        pool = sub[~sub["gene"].isin(all_markers)].copy()
        if pool.empty:
            continue

        n = random_factor * marker_rows["gene"].nunique()
        random_rows = pool.sample(
            n=n,
            replace=n > len(pool),
            random_state=int(rng.integers(0, 2**32 - 1)),
        ).copy()

        marker_rows["gene_set"] = "Marker"
        random_rows["gene_set"] = "Random"
        rows.extend([marker_rows, random_rows])

    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

violin_df = make_marker_random_df(combined_explanations, markers_by_dataset, random_factor=10, seed=1)
groups = violin_df.loc[violin_df["gene_set"] == "Marker", "group"].value_counts()
violin_df = violin_df[violin_df["group"].isin(groups[groups > 3].index)]

if violin_df.empty:
    missing = {
        ds: sorted(set(combined_explanations.loc[combined_explanations["dataset"] == ds, "group"])
                   - set(markers_by_dataset.get(ds, {})))
        for ds in datasets
    }
    raise ValueError(f"No marker/random rows were created. Check cell-type names. Unmatched groups: {missing}")

In [ ]:
markers_palette = {"Marker": colors_to_use[5], "Random": colors_to_use[11]}
datasets_aliases = {"PBMC": "PBMC", "HumanBrain": "Brain", "BMMCMultiOme": "BMMC"}
hue_order = ["Marker", "Random"]
h_order = sorted(combined_explanations["homology_dim"].dropna().unique())
from matplotlib.ticker import MaxNLocator, ScalarFormatter

def format_importance_axis(ax, nbins=4):
    ax.xaxis.set_major_locator(MaxNLocator(nbins=nbins))

    formatter = ScalarFormatter(useMathText=True)
    formatter.set_scientific(True)
    formatter.set_powerlimits((-2, 2))  # use scientific notation for small values
    ax.xaxis.set_major_formatter(formatter)

    ax.tick_params(axis="x", labelrotation=0)
    ax.xaxis.get_offset_text().set_fontsize(12)
    
params = {
    'axes.labelsize': 17,
    'axes.titlesize': 17,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    "lines.linewidth": 3,
    "figure.dpi": 300}
plt.rcParams.update(params)
fig, axes = plt.subplots(3,3, figsize=(16,15))
panel = 0

for i, dataset in enumerate(datasets):
    for j, h in enumerate(h_order):
        ax = axes[i, j]
        sub = violin_df[
            (violin_df["dataset"] == dataset) &
            (violin_df["homology_dim"] == h)
        ].copy()

        if sub.empty:
            ax.set_axis_off()
            panel += 1
            continue

        order = [ct for ct in markers_by_dataset[dataset] if ct in set(sub["group"])]

        sns.violinplot(
            data=sub,
            y="group",
            x="score",
            hue="gene_set",
            order=order,
            hue_order=hue_order,
            palette=markers_palette,
            cut=0,
            inner="quartile",
            scale="width",
            ax=ax,
            orient="h",
        )

        pairs = [
            ((ct, "Marker"), (ct, "Random"))
            for ct in order
            if set(sub.loc[sub["group"] == ct, "gene_set"]) == {"Marker", "Random"}
        ]

        if pairs:
            annotator = Annotator(
                ax,
                pairs,
                data=sub,
                y="group",
                x="score",
                hue="gene_set",
                order=order,
                hue_order=hue_order,
                orient="h",
            )
            annotator.configure(
                test="Mann-Whitney",
                text_format="star",
                loc="outside",
                correction_format="replace",
                comparisons_correction="BH",
                verbose=False,
                show_test_name=False,
            )
            annotator.apply_and_annotate()
            format_importance_axis(ax, nbins=4)

        ax.set_title(
            rf"$\mathbf{{{string.ascii_uppercase[panel]}}}$     H{h} | {datasets_aliases[dataset]}",
            loc="left",
            x=-0.08,
            fontsize=20,
            pad=0,
        )

        ax.set_xlabel("Importance")
        ax.set_ylabel("" if j == 0 else None)
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(False)

        
        if j == 0:
            labels = ["\n".join(tick.get_text().split("_")) for tick in ax.get_yticklabels()]
            ax.set_yticklabels(labels)
        else:
            ax.set_yticks([])
            ax.set_ylabel("")
            
        if ax.get_legend() is not None:
            ax.get_legend().remove()

        panel += 1

handles, labels_ = axes[0, 0].get_legend_handles_labels()
axes[1, -1].legend(handles, labels_,  bbox_to_anchor=(1.01, 0.85),
                    title="", fontsize=17, title_fontsize=20, frameon=False)

fig.tight_layout(h_pad=3)
fig.savefig("Figures/XplainTDA_Figure4.png", dpi=300, bbox_inches="tight", pad_inches=0.3)
plt.show()

# Figure 5

In [ ]:
def natural_key(x):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", str(x))]

def norm_col(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower())

def find_col(columns, candidates):
    cols = list(columns)
    lookup = {norm_col(c): c for c in cols}
    for c in candidates:
        if norm_col(c) in lookup:
            return lookup[norm_col(c)]
    for col in cols:
        n = norm_col(col)
        if any(norm_col(c) in n for c in candidates):
            return col
    return None

def numeric_series(x):
    return pd.to_numeric(
        x.astype(str)
         .str.replace("<", "", regex=False)
         .str.replace(">", "", regex=False)
         .str.replace("%", "", regex=False),
        errors="coerce",
    )

def parse_homer_motif_file(path):
    path = Path(path)
    if not path.exists():
        return None, None, pd.DataFrame(columns=list("ACGT"))

    lines = [l.strip() for l in path.read_text(errors="ignore").splitlines() if l.strip()]
    if not lines:
        return None, None, pd.DataFrame(columns=list("ACGT"))

    header = lines[0].lstrip(">")
    fields = re.split(r"\t+", header)
    consensus = fields[0] if fields else None
    motif_name = fields[1] if len(fields) > 1 else consensus

    pwm = []
    for line in lines[1:]:
        vals = line.split()
        if len(vals) < 4:
            continue
        try:
            pwm.append([float(v) for v in vals[:4]])
        except ValueError:
            pass

    return motif_name, consensus, pd.DataFrame(pwm, columns=list("ACGT"))

def tf_from_motif_name(name):
    name = str(name)
    return re.split(r"[\(/]", name)[0].strip()

def parse_homer_result_block(outdir, result_txt, motif_dir, source, fdr_cutoff=0.05):
    outdir = Path(outdir)
    result_txt = Path(result_txt)
    motif_dir = Path(motif_dir)

    motif_rows = []
    pwm_rows = []

    if not result_txt.exists():
        return motif_rows, pwm_rows

    df = pd.read_csv(result_txt, sep="\t")
    if df.empty:
        return motif_rows, pwm_rows

    name_col = find_col(df.columns, ["Motif Name", "Name", "Motif"])
    consensus_col = find_col(df.columns, ["Consensus", "Motif"])
    p_col = find_col(df.columns, ["P-value", "Pvalue"])
    logp_col = find_col(df.columns, ["Log P-value", "LogP"])
    fdr_col = find_col(df.columns, ["q-value (Benjamini)", "q-value", "qvalue", "FDR"])

    if fdr_col is None:
        return motif_rows, pwm_rows

    df = df.copy()
    df["FDR"] = numeric_series(df[fdr_col])
    df = df[df["FDR"] < fdr_cutoff].reset_index(drop=False).rename(columns={"index": "_original_rank"})

    motif_files = sorted(motif_dir.glob("*.motif"), key=natural_key)

    for _, row in df.iterrows():
        original_rank = int(row["_original_rank"])
        motif_file = motif_files[original_rank] if original_rank < len(motif_files) else None

        file_name, file_consensus, pwm = (
            parse_homer_motif_file(motif_file)
            if motif_file is not None
            else (None, None, pd.DataFrame(columns=list("ACGT")))
        )

        motif_name = row[name_col] if name_col is not None and pd.notna(row[name_col]) else file_name
        motif = row[consensus_col] if consensus_col is not None and pd.notna(row[consensus_col]) else file_consensus
        tf = tf_from_motif_name(motif_name)

        pwm_json = None
        a_w = c_w = g_w = t_w = None

        if not pwm.empty:
            pwm = pwm.astype(float)
            pwm_json = json.dumps(pwm.to_dict(orient="list"))
            a_w = ",".join(map(str, pwm["A"].to_numpy()))
            c_w = ",".join(map(str, pwm["C"].to_numpy()))
            g_w = ",".join(map(str, pwm["G"].to_numpy()))
            t_w = ",".join(map(str, pwm["T"].to_numpy()))

            for pos, vals in pwm.reset_index(drop=True).iterrows():
                pwm_rows.append({
                    "motif_id": None,   # filled later
                    "position": pos + 1,
                    "A": vals["A"],
                    "C": vals["C"],
                    "G": vals["G"],
                    "T": vals["T"],
                })

        motif_rows.append({
            "motif_id": None,  # filled later
            "source": source,
            "motif_name_tf": motif_name,
            "TF": tf,
            "motif": motif,
            "FDR": row["FDR"],
            "p_value": numeric_series(pd.Series([row[p_col]])).iloc[0] if p_col else np.nan,
            "log_p_value": numeric_series(pd.Series([row[logp_col]])).iloc[0] if logp_col else np.nan,
            "motif_file": str(motif_file) if motif_file is not None else None,
            "pwm_json": pwm_json,
            "A_weights": a_w,
            "C_weights": c_w,
            "G_weights": g_w,
            "T_weights": t_w,
        })

    return motif_rows, pwm_rows

def iter_homer_output_dirs(root="Datasets", datasets=None, labels=None, groups=None, hs=None, tops=None):
    root = Path(root)
    datasets = set(datasets) if datasets is not None else None
    labels = set(labels) if labels is not None else None
    groups = set(groups) if groups is not None else None
    hs = set(hs) if hs is not None else None
    tops = set(tops) if tops is not None else None

    pattern = "*/XplainTDA/Peak/Explanations/*/results/groups/Motif/*/H*/top_200"

    for outdir in sorted(root.glob(pattern), key=natural_key):
        if not outdir.is_dir():
            continue

        parts = outdir.parts
        dataset = parts[parts.index("XplainTDA") - 1]
        label = parts[parts.index("Explanations") + 1]
        motif_idx = parts.index("Motif")
        cell_type = parts[motif_idx + 1]
        h = parts[motif_idx + 2]
        top = parts[motif_idx + 3]

        if datasets and dataset not in datasets:
            continue
        if labels and label not in labels:
            continue
        if groups and cell_type not in groups:
            continue
        if hs and h not in hs:
            continue
        if tops and top not in tops:
            continue

        yield dataset, label, cell_type, h, top, outdir

def collect_significant_homer_motifs(
    root="Datasets",
    datasets=("PBMC", "HumanBrain", "BMMCMultiOme"),
    labels=None,
    groups=None,
    hs=None,
    tops=None,
    fdr_cutoff=0.05,
):
    motif_rows = []
    pwm_rows = []

    for dataset, label, cell_type, h, top, outdir in iter_homer_output_dirs(
        root=root,
        datasets=datasets,
        labels=labels,
        groups=groups,
        hs=hs,
        tops=tops,
    ):
        blocks = [
            ("known", outdir / "knownResults.txt", outdir / "knownResults"),
            ("de_novo", outdir / "homerResults.txt", outdir / "homerResults"),
        ]

        for source, result_txt, motif_dir in blocks:
            rows, pwms = parse_homer_result_block(
                outdir=outdir,
                result_txt=result_txt,
                motif_dir=motif_dir,
                source=source,
                fdr_cutoff=fdr_cutoff,
            )

            for row_i, row in enumerate(rows):
                motif_id = f"{dataset}|{label}|{cell_type}|{h}|{top}|{source}|{row_i}"
                row.update({
                    "motif_id": motif_id,
                    "dataset": dataset,
                    "label": label,
                    "cell_type": cell_type,
                    "homology_dim": h,
                    "top_selection": top,
                    "homer_outdir": str(outdir),
                })
                motif_rows.append(row)

                for pwm in pwms:
                    if pwm["motif_id"] is None:
                        pwm["motif_id"] = motif_id

            pwm_rows.extend(pwms)

    motif_table = pd.DataFrame(motif_rows)
    pwm_long = pd.DataFrame(pwm_rows)

    if not motif_table.empty:
        cols = [
            "dataset", "label", "cell_type", "homology_dim", "top_selection",
            "source", "motif_name_tf", "TF", "motif", "FDR", "p_value", "log_p_value",
            "A_weights", "C_weights", "G_weights", "T_weights", "pwm_json",
            "motif_file", "homer_outdir", "motif_id",
        ]
        motif_table = motif_table[[c for c in cols if c in motif_table.columns]]
        motif_table = motif_table.sort_values(
            ["dataset", "cell_type", "homology_dim", "top_selection", "FDR"],
            kind="stable",
        ).reset_index(drop=True)

    return motif_table, pwm_long

In [ ]:
motif_table, pwm_long = collect_significant_homer_motifs(
    root="Datasets",
    datasets=["PBMC", "HumanBrain", "BMMCMultiOme"],
    fdr_cutoff=0.05,
)
motif_table.index=[f"{d}_{ct}_{h}_{tf}" for d, ct, h, tf in zip(motif_table["dataset"],motif_table["cell_type"], motif_table["homology_dim"], motif_table["TF"])]
print(motif_table.shape)
motif_table.to_csv("Tables/HOMER_significant_motifs.tsv", sep="\t")
motif_table.head()

In [ ]:
ok_indeces=["BMMCMultiOme_B_H0_EBF1", "BMMCMultiOme_Progenitor_H0_RUNX1", "BMMCMultiOme_T_H0_ETS1",
           "HumanBrain_Astrocyte_H0_Stat3", "HumanBrain_Microglia_H1_PU.1", "HumanBrain_Purkinje_neuron_H2_RORa",
           "PBMC_DC_H0_SpiB","PBMC_Monocytes_H0_CEBP","PBMC_NK_H0_Eomes"]

restricted_motifs = pd.read_csv("Tables/HOMER_significant_motifs.tsv", sep="\t", index_col=0).loc[ok_indeces]
restricted_motifs

In [ ]:
dataset_order = ["PBMC", "HumanBrain","BMMCMultiOme"]

dataset_aliases = {
    "PBMC": "PBMC",
    "HumanBrain": "Brain",
    "BMMCMultiOme": "BMMC",
}

dna_colors = {
    "A": colors_to_use[19],
    "C": colors_to_use[14],
    "G": colors_to_use[2],
    "T": colors_to_use[5],
}
   
def parse_weights(x):
    if pd.isna(x):
        return []
    return [float(v) for v in str(x).replace(";", ",").split(",") if str(v).strip()]


def pwm_from_weight_columns(row):
    pwm = pd.DataFrame({
        "A": parse_weights(row["A_weights"]),
        "C": parse_weights(row["C_weights"]),
        "G": parse_weights(row["G_weights"]),
        "T": parse_weights(row["T_weights"]),
    })

    if pwm.empty:
        raise ValueError(f"Empty PWM for {row['dataset']} | {row['cell_type']} | {row['TF_canonical']}")

    if not (pwm.shape[0] == len(parse_weights(row["A_weights"])) == len(parse_weights(row["C_weights"])) == len(parse_weights(row["G_weights"])) == len(parse_weights(row["T_weights"]))):
        raise ValueError(f"PWM length mismatch for {row['dataset']} | {row['cell_type']} | {row['TF']}")

    return pwm


def panel_title(row):
    return f"{row['TF']} | {row['cell_type']}\n"

In [ ]:
plt.rcParams.update({
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 300,
})

fig, axes = plt.subplots(
    nrows=3,
    ncols=3,
    figsize=(16, 10),
)
panel=0
for i, dataset in enumerate(dataset_order):
    sub = restricted_motifs[restricted_motifs["dataset"] == dataset].copy()

    # keep the order from the spreadsheet
    sub = sub.reset_index(drop=True)

    for j in range(3):
        ax = axes[i, j]

        if j >= len(sub):
            ax.set_axis_off()
            continue

        row = sub.iloc[j]
        pwm = pwm_from_weight_columns(row)

        logo = logomaker.Logo(
            pwm,
            ax=ax,
            color_scheme=dna_colors,
        )

        logo.style_spines(visible=False)
        logo.style_xticks(fmt="%d", anchor=0)

        ax.set_title(panel_title(row), fontsize=13)
        ax.annotate(rf"$\mathbf{{{string.ascii_uppercase[panel]}}}$", (0, 1.1), xycoords="axes fraction", fontsize=20)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.spines[["top", "right", "left", "bottom"]].set_visible(False)

        if j == 0:
            ax.text(
                -0.22,
                0.5,
                dataset_aliases.get(dataset, dataset),
                transform=ax.transAxes,
                rotation=90,
                va="center",
                ha="center",
                fontsize=18,
            )
        panel+=1

fig.tight_layout(w_pad=2, h_pad=2)
fig.savefig("Figures/XplainTDA_Figure5.png", dpi=300, bbox_inches="tight")
plt.show()